# Extract reproducibility of trace element values using duplicate LAICPMS values

Author: **Niels J. de Winter** (*n.j.de.winter@vu.nl*)<br>
Assistant Professor Vrije Universiteit Amsterdam

In [1]:
# Load functions and libraries
import pandas as pd
import numpy as np

## Extract specimen names

In [2]:
# Load All data summary to extract specimen names for subsetting
data_summary = pd.read_csv("All_data_summary.csv")
# Extract unique specimen names from the summary data and combine with first 3 letters of treatment information to create a unique identifier for each specimen
specimen_names = np.unique(data_summary['specimen'].astype(str) + '_' + data_summary['treatment'].str[:3])
print(specimen_names)

['B255_int' 'B323_int' 'B325_int' 'G652_sub' 'G664_sub' 'G668_sub'
 'G691_int' 'G717_sub' 'G719_int' 'G723_sub' 'G728_int' 'G730_sub'
 'G732_sub' 'G737_sub' 'G753_sub' 'G767_sub' 'G770_sub' 'G779_int'
 'G781_sub']


## Loop through files and extract information about uncertainty using duplicate analyses

In [3]:
specimen = specimen_names[0]  # Example: select the first specimen for subsetting
print(f"Selected specimen for subsetting: {specimen}")
dat = pd.read_csv(specimen + '.csv') # Load the data for the current specimen
dat.head()

Selected specimen for subsetting: B255_int


,time,depth,Xpos,Ypos,Ca43,23Na/43Ca,25Mg/43Ca,43Ca/43Ca,55Mn/43Ca,88Sr/43Ca,138Ba/43Ca
0,s,mm,,,cps,mmol/mol,mmol/mol,mmol/mol,mmol/mol,mmol/mol,mmol/mol
1,62.33211,0.000445779,30944.1062,80703.2753,6301.588,199.0592481,1.253571207,1000,0.306491077,0.013676373,0
2,62.44125,0.000891558,30944.46978,80703.01737,5301.123838,235.4822349,3.406310372,1000,0.347474326,0,0
3,62.5504,0.001337336,30944.83337,80702.75945,4600.846556,284.3316789,1.771388245,1000,0.380053324,0.01932572,0
4,62.65955,0.001783115,30945.19695,80702.50152,5751.322804,221.3077633,3.115196703,1000,0.348871381,0.037762937,0


In [5]:
# Create dataframe for storing information about reproduciblity per specimen and per element
reproducibility_df = pd.DataFrame(columns=['specimen', 'element', 'combined_std_dev'])

# List of elements to analyze
elements = ['23Na/43Ca', '25Mg/43Ca', '55Mn/43Ca', '88Sr/43Ca', '138Ba/43Ca']

for specimen in specimen_names:
    dat = pd.read_csv(specimen + '.csv') # Load the data for the current specimen
    dat = dat[1:].reset_index(drop=True) # remove top row that contains string values instead of numeric values
    dat = dat.apply(pd.to_numeric, errors='coerce') # convert all columns to numeric values, coercing errors to NaN
    unique_x, indices, counts = np.unique(dat['depth'], return_inverse = True, return_counts = True) # Find unique x-values and their indices
    print(f'{len(dat["depth"]) - len(unique_x)} duplicates were identified in specimen {specimen}')

    # Loop through each element and calculate the combined standard deviation for the duplicates
    for element in elements:
        element_data = dat[element].copy()  # Assuming the proxy values are in the specified columns
        # Convert zeroes to NaN to avoid skewing the standard deviation calculation
        element_data[element_data == 0] = np.nan
        # Calculate the mean uncertainty (standard deviation) based on the duplicates
        std_devs = np.zeros(len(unique_x))
        for i in range(len(unique_x)):
            duplicate_indices = np.where(indices == i)[0]
            if len(duplicate_indices) > 1:
                std_devs[i] = np.std(element_data.iloc[duplicate_indices])  # Calculate standard deviation for duplicates

        # Calculate the combined standard deviation
        combined_variance = np.sum(std_devs[std_devs > 0]**2)  # Sum of variances
        combined_std_dev = np.sqrt(combined_variance / np.sum(counts > 1))  # Combined standard deviation
        print(f'Combined standard deviation of element {element} in specimen {specimen} from duplicates: {combined_std_dev:.3g} mmol/mol')
        reproducibility_df.loc[len(reproducibility_df)] = {
            'specimen': specimen,
            'element': element,
            'combined_std_dev': combined_std_dev
        }
# Save the reproducibility results to a CSV file
print(reproducibility_df.head())
reproducibility_df.to_csv('TE_duplicate_reproducibility_results.csv', index=False)

156 duplicates were identified in specimen B255_int
Combined standard deviation of element 23Na/43Ca in specimen B255_int from duplicates: 4.34 mmol/mol
Combined standard deviation of element 25Mg/43Ca in specimen B255_int from duplicates: 0.164 mmol/mol
Combined standard deviation of element 55Mn/43Ca in specimen B255_int from duplicates: 0.00134 mmol/mol
Combined standard deviation of element 88Sr/43Ca in specimen B255_int from duplicates: 0.471 mmol/mol
Combined standard deviation of element 138Ba/43Ca in specimen B255_int from duplicates: 0.000696 mmol/mol
115 duplicates were identified in specimen B323_int
Combined standard deviation of element 23Na/43Ca in specimen B323_int from duplicates: 4.9 mmol/mol
Combined standard deviation of element 25Mg/43Ca in specimen B323_int from duplicates: 0.513 mmol/mol
Combined standard deviation of element 55Mn/43Ca in specimen B323_int from duplicates: 0.00402 mmol/mol
Combined standard deviation of element 88Sr/43Ca in specimen B323_int from 